# Global-mean surface climate drift against observations — Full-CPL spin-up + piControl and alternate spin-ups

How far the global-mean surface climate is from the observed pre-industrial state as the spin-up proceeds: for each
field, annual global means of the run against the observed annual global means of `OBS_YEARS` (NOAA-20C reanalysis,
1871–1890), over a sliding window of the same length (20 yr):
* **bias** = window mean − observed mean (grey band: ±2 standard errors of the observed mean)
* **RMSE** = root-mean-square difference of the window's annual means and the observed annual means (bias and
  interannual mismatch together)

Fields: TS, TREFHT, PSL, PRECT, SHFLX, LHFLX (= L_v·QFLX), FLUT, from the `post/time_series/atm` area means.
The reference is a reanalysis: its turbulent fluxes and OLR carry their own biases (the near-constant FLUT and
SHFLX offsets largely reflect these), so the drift — the change of the bias with time — is the robust signal.
Full-CPL: spin-up years 1–2000 followed by its piControl (drawn as 2001–2500). FC-FOSI-S1/S2: years 1–350; their
atmosphere output exists only for the fully coupled segments, so their curves start once a full window of coupled
years is available.

Figures (`<metric>` = bias, rmse): `atm_drift_obs_<metric>_0-2500.pdf` (all runs) and
`atm_drift_obs_<metric>_0-350.pdf` (first 350 years). Figures go to `FIG_DIR` (public_html), caches to `DATA_DIR`.

In [ ]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
%matplotlib inline

import matplotlib.pyplot as plt

import numpy as np
import xarray as xr

from util.experiments import make_run_dict
from util.figures import publish
from util.area_mean_drift import load_or_read
from util.obs_drift import obs_annual_global_mean, model_annual_global_mean, sliding_drift, plot_drift

## Setup

In [ ]:
# ---- paths ----
MODEL_ROOT   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"     # case directories
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/atm_drift_obs"
DATA_DIR     = f"{OUTPUT_ROOT}/data/atm_drift_obs"         # cached/derived data (index: data/README.md)

OBS_DIR      = "/lcrc/soft/climate/e3sm_diags_data/obs_for_e3sm_diags/time-series/NOAA-20C"   # e3sm_diags obs time series

# ---- experiments: case directory names (under MODEL_ROOT) and model-year periods ----
EXPERIMENTS = {
    "Full-CPL": {
        "spin-up":   {"name": "20231209.v3.LR.piControl-spinup.chrysalis",      "period": "0001-2000"},
        "piControl": {"name": "v3.LR.piControl",                                "period": "0001-0500"},
    },
    "AltBG1": {
        "spin-up":   {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG2": {
        "spin-up":   {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
}
LABEL = {"Full-CPL": "FC", "AltBG1": "FC-FOSI-S1", "AltBG2": "FC-FOSI-S2", "AltBG3": "FC-FOSI-S3"}

# ---- analysis settings ----
EXPS        = ["Full-CPL", "AltBG1", "AltBG2"]   # runs shown
COLORS      = ["black", "#cc0000", "#377eb8"]    # one per run
SPINUP_YEARS = {"Full-CPL": (1, 2000), "AltBG1": (1, 350), "AltBG2": (1, 350)}   # years read from each spin-up case
CTRL_YEARS  = (1, 500)      # Full-CPL piControl years, drawn after its spin-up
OBS_YEARS   = (1871, 1890)  # observed reference period (window length = its number of years)
MARKERS     = [221, 2000]   # vertical lines: first fully coupled year of FC-FOSI, end of the Full-CPL spin-up

In [ ]:
# Shared inputs/outputs
run_dict = make_run_dict(EXPERIMENTS, MODEL_ROOT)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

## 1 Annual global means (model and observations)
### Settings
`FIELDS`: model variable -> observed variable in `OBS_DIR/<obs>_183601_201512.nc`, scale applied to it, unit, title.

In [ ]:
FIELDS = {
    "TS":     dict(obs="ts",   scale=1.0,     unit="K",              title="Surface temperature"),
    "TREFHT": dict(obs="tas",  scale=1.0,     unit="K",              title="Surface air temperature"),
    "PSL":    dict(obs="psl",  scale=0.01,    unit="hPa",            title="Sea-level pressure"),
    "PRECT":  dict(obs="pr",   scale=86400.0, unit=r"mm day$^{-1}$", title="Precipitation"),
    "SHFLX":  dict(obs="hfss", scale=1.0,     unit=r"W m$^{-2}$",    title="Sensible heat flux"),
    "LHFLX":  dict(obs="hfls", scale=1.0,     unit=r"W m$^{-2}$",    title="Latent heat flux"),
    "FLUT":   dict(obs="rlut", scale=1.0,     unit=r"W m$^{-2}$",    title="Outgoing longwave radiation"),
}
READ_VARS = ["TS", "TREFHT", "PSL", "SHFLX", "FLUT"]   # read directly; PRECT, LHFLX are derived
DERIVED   = ("PRECT", "LHFLX")

### Compute

In [ ]:
obs = {}
for v, cfg in FIELDS.items():
    obs[v] = obs_annual_global_mean(f"{OBS_DIR}/{cfg['obs']}_183601_201512.nc", cfg["obs"], OBS_YEARS, scale=cfg["scale"])
    print(f"obs {v:7s} {float(obs[v].mean()):9.3f} ± {float(obs[v].std()):.3f} {cfg['unit']}")
xr.Dataset({v: o for v, o in obs.items()}).to_netcdf(os.path.join(DATA_DIR, f"obs_NOAA-20C_global_annual_{OBS_YEARS[0]}-{OBS_YEARS[1]}.nc"))

def annual(case, years, tag):
    cache = os.path.join(DATA_DIR, f"{tag}_area_mean_monthly_{years[0]:04d}-{years[1]:04d}.nc")
    da = load_or_read(cache, os.path.join(MODEL_ROOT, case, "post/time_series/atm"), READ_VARS, years, derived=DERIVED)
    return {v: model_annual_global_mean(da, v) for v in FIELDS}

model = {}
for t in EXPS:
    model[t] = annual(run_dict[t]["spin-up"]["name"], SPINUP_YEARS[t], t)
    ctrl = run_dict[t]["piControl"]["name"]
    if ctrl != run_dict[t]["spin-up"]["name"]:                       # Full-CPL: append its piControl
        pc = annual(ctrl, CTRL_YEARS, f"{t}_piControl")
        off = SPINUP_YEARS[t][1]
        model[t] = {v: xr.concat([model[t][v], pc[v].assign_coords(year=pc[v]["year"] + off)], "year")
                    for v in FIELDS}

drift = {v: {} for v in FIELDS}
for t, color in zip(EXPS, COLORS):
    for v in FIELDS:
        drift[v][LABEL[t]] = dict(ds=sliding_drift(model[t][v], obs[v]), color=color)
n = OBS_YEARS[1] - OBS_YEARS[0] + 1
obs_std = {v: dict(sem=float(obs[v].std()) / np.sqrt(n)) for v in FIELDS}
publish(DATA_DIR)

## 2 Figures
### Settings

In [ ]:
DRIFT_STYLE = dict(ncols=2, fontz=12.0, panel_letters=True)
XRANGES     = {"0-2500": (0, 2500), "0-350": (0, 350)}   # one figure per range and metric

### Plot

In [ ]:
fields = {v: (cfg["unit"], cfg["title"]) for v, cfg in FIELDS.items()}
for metric in ("bias", "rmse"):
    for tag, xlim in XRANGES.items():
        fig, axes = plot_drift(drift, fields, metric=metric, obs_std=obs_std,
                               markers=[m for m in MARKERS if xlim[0] < m < xlim[1]], **DRIFT_STYLE)
        for ax in axes.flat:
            ax.set_xlim(*xlim)
        out = os.path.join(FIG_DIR, f"atm_drift_obs_{metric}_{tag}.pdf")
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))

publish(FIG_DIR, DATA_DIR)